In [1]:
import os

In [2]:
%pwd

'c:\\Users\\hulkh\\Desktop\\Python\\supplier_clustering\\Supplier_clustering\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\hulkh\\Desktop\\Python\\supplier_clustering\\Supplier_clustering'

In [17]:
import pandas as pd
data = pd.read_csv("artifacts/data_ingestion/purchase_orders.csv")
data.head()

,po_id,order_date,promised_delivery_date,actual_delivery_date,supplier_id,supplier_name,supplier_country,category,item,business_unit,unit_price,quantity,line_total,payment_terms,on_contract,quality_rejected
0,PO-2023-00441,2023-01-01,2023-01-09,2023-01-11,SUP-038,Bergmann MRO Services,Germany,MRO,Safety Gloves (per 100),Manufacturing,102.23,22,2249.06,Net 45,True,False
1,PO-2023-00598,2023-01-01,2023-01-27,2023-01-27,SUP-053,Sterling Shipping Lines,USA,Logistics,Ocean Container (40ft),Retail Operations,3271.58,2,6543.16,Net 30,True,False
2,PO-2023-00300,2023-01-01,2023-01-09,2023-01-09,SUP-022,Redwood Components,USA,Electronics,Power Supply Unit 450W,Retail Operations,60.25,179,10784.75,Net 15,True,False
3,PO-2023-00169,2023-01-01,2023-01-09,2023-01-09,SUP-016,Great Wall Containers,China,Packaging,Wooden Pallets,Manufacturing,15.62,247,3858.14,Net 15,True,False
4,PO-2023-00280,2023-01-01,2023-02-02,2023-02-02,SUP-036,Golden Dragon Components,China,Electronics,Connector Kit CK-8,Corporate,4.30,927,3986.10,Net 45,True,False


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47128 entries, 0 to 47127
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   po_id                   47128 non-null  object 
 1   order_date              47128 non-null  object 
 2   promised_delivery_date  47128 non-null  object 
 3   actual_delivery_date    47128 non-null  object 
 4   supplier_id             47128 non-null  object 
 5   supplier_name           47128 non-null  object 
 6   supplier_country        47128 non-null  object 
 7   category                47128 non-null  object 
 8   item                    47128 non-null  object 
 9   business_unit           47128 non-null  object 
 10  unit_price              47128 non-null  float64
 11  quantity                47128 non-null  int64  
 12  line_total              47128 non-null  float64
 13  payment_terms           47128 non-null  object 
 14  on_contract             47128 non-null

In [7]:
data.isnull().sum()

po_id                     0
order_date                0
promised_delivery_date    0
actual_delivery_date      0
supplier_id               0
supplier_name             0
supplier_country          0
category                  0
item                      0
business_unit             0
unit_price                0
quantity                  0
line_total                0
payment_terms             0
on_contract               0
quality_rejected          0
dtype: int64

In [8]:
# entity

from dataclasses import dataclass
from pathlib import Path

#entity
@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    unzip_data_dir: Path
    all_schema: dict

In [9]:
# configuration manager
from src.Supplier_clustering.constants import *
from src.Supplier_clustering.utils.common import read_yaml, create_directories

In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir = config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

In [11]:
import os
from src.Supplier_clustering.utils import logger

In [12]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config


    def validate_columns(self, data: pd.DataFrame) -> bool:
        """
        Validate that:
        1. No expected columns are missing.
        2. No unexpected columns are present.
        """

        expected_columns = set(self.config.all_schema.keys())
        actual_columns = set(data.columns)

        missing_columns = expected_columns - actual_columns
        extra_columns = actual_columns - expected_columns

        validation_status = True

        if missing_columns:
            print(f"Missing columns: {missing_columns}")
            validation_status = False

        if extra_columns:
            print(f"Unexpected columns: {extra_columns}")
            validation_status = False

        return validation_status

    def validate_datatypes(self, data: pd.DataFrame) -> bool:
        """
        Validate that the dataframe columns have
        the expected datatypes defined in schema.yaml.
        """

        validation_status = True

        for column, expected_dtype in self.config.all_schema.items():

            if column in data.columns:

                actual_dtype = str(data[column].dtype)

                if actual_dtype != expected_dtype:
                    print(
                        f"Datatype mismatch for '{column}': "
                        f"expected {expected_dtype}, "
                        f"got {actual_dtype}"
                    )

                    validation_status = False

        return validation_status

    def validate_all_columns(self) -> bool:
        """
        Run all data validation checks.
        """

        try:
            data = pd.read_csv(self.config.unzip_data_dir)

            columns_valid = self.validate_columns(data)

            datatypes_valid = self.validate_datatypes(data)

            validation_status = columns_valid and datatypes_valid

            with open(self.config.STATUS_FILE, "w") as f:
                f.write(f"Validation status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [16]:
#pipeline
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

[2026-08-22 15:45:49,727: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-22 15:45:49,728: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-22 15:45:49,732: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-08-22 15:45:49,734: INFO: common: created directory at: artifacts]
[2026-08-22 15:45:49,735: INFO: common: created directory at: artifacts/data_validation]
